# CollatorSentenceTransformer
## Padding-free inference with `DataCollatorWithFlattening` + Flash Attention varlen

### The problem

Standard `SentenceTransformer.encode()` pads all sequences in a batch to the **longest** sequence's length.  
For variable-length inputs this wastes memory and compute — a batch of 31 short sentences and one long sentence
pays full cost for the long one.

```
Standard (padded):          Collator (packed):
Sentence 1  [▓▓▓░░░░░]     Sentence 1  [▓▓▓]
Sentence 2  [▓▓▓▓▓░░░]  →  Sentence 2  [▓▓▓▓▓]
Sentence 3  [▓▓▓▓▓▓▓▓]     Sentence 3  [▓▓▓▓▓▓▓▓]
            ░ = wasted pad
```

### The solution

`DataCollatorWithFlattening` (transformers ≥ 5.0) concatenates all sequences into one flat tensor and provides
`cu_seqlens` (cumulative sequence lengths) so flash attention's **varlen kernel** can still prevent cross-sequence
attention while skipping all padding.

### Requirements
- `flash_attn` installed (`pip install flash-attn --no-build-isolation`)
- Model that supports `flash_attention_2` — this notebook uses **`ibm-granite/granite-embedding-97m-multilingual-r2`** (ModernBERT architecture)
- CUDA GPU

In [ ]:
import inspect
import time
import warnings

import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import DataCollatorWithFlattening

from sentence_transformers import SentenceTransformer

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

try:
    import flash_attn
    print("flash_attn:", flash_attn.__version__)
except ImportError:
    raise ImportError("flash_attn not found — install with: pip install flash-attn --no-build-isolation")

## `CollatorSentenceTransformer` class

Inherits from `SentenceTransformer` and overrides `encode()` to:

1. Tokenise each sentence **individually** (no padding).
2. Pack all sentences in the batch into one flat tensor via `DataCollatorWithFlattening`.
3. Build model inputs by mapping collator output keys to the model's expected argument names
   (ModernBERT uses `cu_seqlens` / `max_seqlen`; other architectures may differ).
4. Run a **single forward pass** over the packed sequence.
5. Slice the output back into per-sentence hidden states and pool.
6. Normalise.

**Pooling mode** (CLS vs mean) is detected by reading `1_Pooling/config.json` directly from the model
repository. Reading from `self[1].get_config_dict()` is not reliable here because `SentenceTransformer`
subclasses go through a different internal loading path (`_load_converted_modules`) that ignores the
model's own `sentence_bert_config.json` and falls back to the library default ('mean', `include_prompt=True`).

In [ ]:
import json
import os

from huggingface_hub import hf_hub_download


def _detect_pooling_mode(model_name_or_path: str) -> str:
    """Read pooling mode from 1_Pooling/config.json in the model repository.

    Handles both the legacy boolean-flag format and the modern 'pooling_mode' key.
    Falls back to 'mean' if the file is absent or unreadable.
    """
    _LEGACY_MAP = {
        "pooling_mode_cls_token":           "cls",
        "pooling_mode_mean_tokens":          "mean",
        "pooling_mode_max_tokens":           "max",
        "pooling_mode_mean_sqrt_len_tokens": "mean_sqrt_len_tokens",
        "pooling_mode_weightedmean_tokens":  "weightedmean",
        "pooling_mode_lasttoken":            "lasttoken",
    }
    # Local model directory
    local = os.path.join(model_name_or_path, "1_Pooling", "config.json")
    if os.path.isfile(local):
        config_path = local
    else:
        try:
            config_path = hf_hub_download(model_name_or_path, "1_Pooling/config.json")
        except Exception:
            return "mean"

    with open(config_path) as f:
        config = json.load(f)

    if "pooling_mode" in config:
        return config["pooling_mode"]
    for key, mode in _LEGACY_MAP.items():
        if config.get(key, False):
            return mode
    return "mean"


class CollatorSentenceTransformer(SentenceTransformer):
    """SentenceTransformer that eliminates padding via DataCollatorWithFlattening.

    Drop-in replacement for SentenceTransformer.encode() for text inputs.
    Requires flash_attn and a model that supports flash_attention_2.

    Parameters
    ----------
    model_name_or_path : str
        HuggingFace model name or local path (same as SentenceTransformer).
    **kwargs
        Forwarded to SentenceTransformer.__init__.
        ``model_kwargs`` defaults to ``{dtype: bfloat16, attn_implementation: flash_attention_2}``
        if not provided.
    """

    # Maps DataCollatorWithFlattening output keys to architecture-specific names.
    # Keys are listed in preference order; the first match found in the model's
    # forward signature wins.
    _COLLATOR_KEY_CANDIDATES: dict[str, list[str]] = {
        "cu_seq_lens_q": ["cu_seq_lens_q", "cu_seqlens"],
        "cu_seq_lens_k": ["cu_seq_lens_k"],
        "max_length_q":  ["max_length_q",  "max_seqlen"],
        "seq_idx":       ["seq_idx",        "indices"],
        "position_ids":  ["position_ids"],
    }

    def __init__(self, model_name_or_path: str, **kwargs) -> None:
        model_kwargs = dict(kwargs.pop("model_kwargs", None) or {})
        model_kwargs.setdefault("dtype", torch.bfloat16)
        model_kwargs.setdefault("attn_implementation", "flash_attention_2")
        super().__init__(model_name_or_path, model_kwargs=model_kwargs, **kwargs)

        self._pack_collator = DataCollatorWithFlattening(
            return_flash_attn_kwargs=True,
            return_seq_idx=True,
            return_position_ids=True,
        )

        # Read pooling mode from 1_Pooling/config.json.  We cannot use
        # self[1].get_config_dict() because ST subclasses go through
        # _load_converted_modules which ignores sentence_bert_config.json
        # and always constructs Pooling with the library default ('mean').
        self._pooling_mode: str = _detect_pooling_mode(model_name_or_path)
        print(f"  pooling_mode: {self._pooling_mode}")

        # Build a key map: collator output key → model forward argument name.
        fwd_params = set(
            inspect.signature(self._first_module().auto_model.forward).parameters.keys()
        )
        self._key_map: dict[str, str] = {}
        for collator_key, candidates in self._COLLATOR_KEY_CANDIDATES.items():
            for candidate in candidates:
                if candidate in fwd_params:
                    self._key_map[collator_key] = candidate
                    break
        print(f"  key_map: {self._key_map}")
        print(f"  extra model params: batch_size={'batch_size' in fwd_params}, "
              f"seq_len={'seq_len' in fwd_params}")
        self._fwd_params = fwd_params

    @torch.inference_mode()
    def encode(
        self,
        sentences: list[str] | str,
        batch_size: int = 32,
        show_progress_bar: bool = False,
        normalize_embeddings: bool = True,
        convert_to_numpy: bool = True,
        convert_to_tensor: bool = False,
        device: str | None = None,
        **_kwargs,
    ) -> np.ndarray | torch.Tensor:
        """Encode sentences using packed (no-padding) flash attention."""
        if isinstance(sentences, str):
            sentences = [sentences]

        target_device = torch.device(device or self.device)
        transformer   = self._first_module()
        tokenizer     = self.tokenizer
        model         = transformer.auto_model
        max_length    = transformer.max_seq_length or 512

        all_embeddings: list[torch.Tensor] = []
        batch_iter = range(0, len(sentences), batch_size)
        if show_progress_bar:
            batch_iter = tqdm(batch_iter, desc="Encoding (collator)")

        for start in batch_iter:
            batch_texts = sentences[start : start + batch_size]

            # ── Step 1: tokenise individually, no padding ──────────────────
            encoded = [
                tokenizer(
                    text,
                    return_attention_mask=False,
                    truncation=True,
                    max_length=max_length,
                )
                for text in batch_texts
            ]
            lengths = [len(e["input_ids"]) for e in encoded]

            # ── Step 2: pack with DataCollatorWithFlattening ────────────────
            packed = self._pack_collator(
                [{"input_ids": e["input_ids"]} for e in encoded]
            )

            # ── Step 3: build model inputs ──────────────────────────────────
            model_inputs: dict = {}

            for src_key, tensor in packed.items():
                dst_key = self._key_map.get(src_key, src_key)
                if dst_key in self._fwd_params:
                    model_inputs[dst_key] = (
                        tensor.to(target_device)
                        if isinstance(tensor, torch.Tensor)
                        else tensor
                    )

            # ModernBERT varlen mode: input_ids must be 1-D (unpadded tokens).
            # The embedding layer accepts 1-D input; attention then has shape
            # [total_tokens, hidden_size] so bs == total_tokens, which matches
            # what flash_attn_varlen_qkvpacked_func returns.
            if "input_ids" in model_inputs:
                model_inputs["input_ids"] = model_inputs["input_ids"].squeeze(0)
            if "position_ids" in model_inputs:
                # Keep [1, total_tokens] so downstream positional ops work.
                model_inputs["position_ids"] = (
                    model_inputs["position_ids"].squeeze(0).unsqueeze(0)
                )

            # ModernBERT expects batch_size and seq_len as plain ints.
            if "batch_size" in self._fwd_params:
                model_inputs["batch_size"] = len(batch_texts)
            if "seq_len" in self._fwd_params:
                model_inputs["seq_len"] = sum(lengths)

            # Pass a non-None attention_mask to bypass the padding-token
            # warning check inside the model, which requires 2-D input_ids.
            if "attention_mask" in self._fwd_params:
                model_inputs["attention_mask"] = torch.zeros(0, device=target_device)

            # ── Step 4: forward pass ────────────────────────────────────────
            outputs = model(**model_inputs)
            hidden = outputs.last_hidden_state  # [total_tokens, H] or [1, total_tokens, H]
            if hidden.dim() == 3:
                hidden = hidden.squeeze(0)      # → [total_tokens, H]

            # ── Step 5: pool each sentence ──────────────────────────────────
            embeddings: list[torch.Tensor] = []
            offset = 0
            for length in lengths:
                if self._pooling_mode == "cls":
                    emb = hidden[offset].float()                      # CLS = first token
                else:
                    emb = hidden[offset : offset + length].float().mean(0)
                embeddings.append(emb)
                offset += length

            batch_embs = torch.stack(embeddings)
            if normalize_embeddings:
                batch_embs = torch.nn.functional.normalize(batch_embs, p=2, dim=1)
            all_embeddings.append(batch_embs)

        result = torch.cat(all_embeddings, dim=0)
        if convert_to_tensor:
            return result
        return result.cpu().numpy()


print("CollatorSentenceTransformer defined.")

## Load the model

In [ ]:
MODEL_NAME = "ibm-granite/granite-embedding-97m-multilingual-r2"

print("Loading CollatorSentenceTransformer...")
cst = CollatorSentenceTransformer(MODEL_NAME)
print(f"\nLoaded on: {cst.device}")

# Also load a standard SentenceTransformer for comparison
print("\nLoading standard SentenceTransformer (bfloat16, flash_attention_2)...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    st = SentenceTransformer(
        MODEL_NAME,
        model_kwargs={"dtype": torch.bfloat16, "attn_implementation": "flash_attention_2"},
    )
print("Done.")

## Basic usage

In [ ]:
sample_texts = [
    "What is the capital of France?",
    "Machine learning models learn patterns from data.",
    "Short.",
    "This is a considerably longer sentence that contains many more tokens than the previous ones, "
    "used here to illustrate the padding overhead that the collator approach eliminates.",
]

embs = cst.encode(sample_texts)
print("Output shape:", embs.shape)
print("Embedding norms:", np.linalg.norm(embs, axis=1))  # should be ~1.0 (normalised)

## Correctness check

Compare `CollatorSentenceTransformer` embeddings against the standard `SentenceTransformer`.
Both use the same weights; the only difference is *how* sequences are packed and how attention is applied.

Expected cosine similarity: **> 0.995** (small residual from bfloat16 rounding — varlen flash
attention computes in a different tile order than padded flash attention, accumulating rounding
errors differently even with identical weights).

In [ ]:
from sentence_transformers import util as st_util

# Generate a diverse set of texts with varying lengths
rng = np.random.default_rng(42)

short_texts = [
    "Hello world", "Quick brown fox", "AI is transforming industries",
    "Retrieval augmented generation", "Vector search at scale",
]
medium_texts = [
    "Large language models have fundamentally changed how we approach NLP tasks.",
    "Embedding models map variable-length text into fixed-size dense vectors.",
    "Flash attention reduces memory complexity from O(n²) to O(n) by computing attention in tiles.",
]
long_texts = [
    "The granite-embedding-97m-multilingual-r2 model is built on the ModernBERT architecture, "
    "which uses rotary position embeddings and flash attention to support sequences of up to 8192 tokens. "
    "The multilingual variant was trained on text from over 100 languages, making it suitable for "
    "cross-lingual retrieval and semantic search applications.",
    "DataCollatorWithFlattening is a utility introduced in transformers 5.0 that concatenates all "
    "sequences in a batch into a single flat tensor, paired with cumulative sequence length tensors "
    "(cu_seqlens) that tell flash attention where each sequence begins and ends, thereby preserving "
    "the invariant that tokens only attend within their own sequence while eliminating padding tokens.",
]

verify_texts = short_texts + medium_texts + long_texts
print(f"Verifying on {len(verify_texts)} texts")

std_embs  = st.encode(verify_texts,  normalize_embeddings=True, convert_to_numpy=True)
coll_embs = cst.encode(verify_texts, normalize_embeddings=True, convert_to_numpy=True)

cos_sims = np.einsum('ij,ij->i', std_embs, coll_embs)   # row-wise dot product of unit vectors
l2_diffs = np.linalg.norm(std_embs - coll_embs, axis=1)

print(f"\nCosine similarity (std vs collator):")
print(f"  mean = {cos_sims.mean():.6f}")
print(f"  min  = {cos_sims.min():.6f}")
print(f"  max  = {cos_sims.max():.6f}")
print(f"\nL2 distance:")
print(f"  mean = {l2_diffs.mean():.6e}")
print(f"  max  = {l2_diffs.max():.6e}")

## Throughput benchmark

The collator approach pays off most when batches contain **variable-length** inputs, because padding waste is highest then.
We benchmark both encoders on a workload of 512 texts with lengths drawn from a realistic distribution (mix of short queries and long passages).

In [ ]:
import random

def make_variable_length_texts(n: int, seed: int = 0) -> list[str]:
    """Generate n texts with lengths sampled from a log-normal distribution."""
    random.seed(seed)
    word = "token "  # simple stand-in; real words wouldn't change the length profile
    lengths = np.clip(
        np.random.default_rng(seed).lognormal(mean=4.5, sigma=1.2, size=n).astype(int),
        2, 400,
    )
    return [word * int(L) for L in lengths]


N_TEXTS   = 512
N_WARMUP  = 64
BATCH_SIZES = [8, 16, 32, 64]

all_texts = make_variable_length_texts(N_TEXTS + N_WARMUP, seed=42)
warmup_texts = all_texts[:N_WARMUP]
bench_texts  = all_texts[N_WARMUP:]

tok_lengths = [len(cst.tokenizer(t, truncation=True, max_length=512)['input_ids'])
               for t in bench_texts]
print(f"Token length distribution over {N_TEXTS} texts:")
print(f"  mean={np.mean(tok_lengths):.1f}  median={np.median(tok_lengths):.1f}  "
      f"max={np.max(tok_lengths)}  min={np.min(tok_lengths)}")
print(f"  (padding waste at batch_size=32: "
      f"~{100*(1 - np.mean(tok_lengths)/np.max(tok_lengths)):.0f}% of tokens are padding in the worst batch)")

In [ ]:
def benchmark(encoder, texts: list[str], batch_size: int, label: str,
              n_repeats: int = 3) -> dict:
    """Run encoder n_repeats times and return throughput stats."""
    times = []
    for _ in range(n_repeats):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = encoder.encode(texts, batch_size=batch_size, normalize_embeddings=True)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return {
        "label":       label,
        "batch_size":  batch_size,
        "n_texts":     len(texts),
        "total_s":     np.mean(times),
        "throughput":  len(texts) / np.mean(times),
        "latency_ms":  np.mean(times) / len(texts) * 1000,
    }


# Warmup both encoders to stabilise GPU clocks
print("Warming up...")
_ = st.encode(warmup_texts,  batch_size=32, normalize_embeddings=True)
_ = cst.encode(warmup_texts, batch_size=32, normalize_embeddings=True)
print("Warmup done.")

results = []
for bs in BATCH_SIZES:
    print(f"batch_size={bs}: ", end="", flush=True)
    r_std  = benchmark(st,  bench_texts, bs, "Standard ST")
    r_coll = benchmark(cst, bench_texts, bs, "Collator ST")
    results.extend([r_std, r_coll])
    speedup = r_coll["throughput"] / r_std["throughput"]
    print(f"standard={r_std['throughput']:.0f} sents/s  collator={r_coll['throughput']:.0f} sents/s  "
          f"speedup={speedup:.2f}x")

print("\nDone.")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

std_rows  = [r for r in results if r["label"] == "Standard ST"]
coll_rows = [r for r in results if r["label"] == "Collator ST"]

bs_vals      = [r["batch_size"] for r in std_rows]
std_tput     = [r["throughput"] for r in std_rows]
coll_tput    = [r["throughput"] for r in coll_rows]
speedups     = [c / s for c, s in zip(coll_tput, std_tput)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

x = np.arange(len(bs_vals))
w = 0.35

ax1.bar(x - w/2, std_tput,  w, label="Standard (padded)", color="#E74C3C", alpha=0.85)
ax1.bar(x + w/2, coll_tput, w, label="Collator (packed)",  color="#2ECC71", alpha=0.85)
ax1.set_xlabel("Batch size");  ax1.set_ylabel("Throughput (sentences/s)")
ax1.set_title("Throughput comparison");  ax1.set_xticks(x);  ax1.set_xticklabels(bs_vals)
ax1.legend();  ax1.grid(axis="y", alpha=0.4)

bars = ax2.bar(x, speedups, 0.5, color="#3498DB", alpha=0.85)
for bar, sp in zip(bars, speedups):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
             f"{sp:.2f}x", ha="center", va="bottom", fontweight="bold")
ax2.axhline(1.0, color="red", linestyle="--", lw=1.5, label="Baseline 1×")
ax2.set_xlabel("Batch size");  ax2.set_ylabel("Speedup (Collator / Standard)")
ax2.set_title("Speedup of Collator over Standard");  ax2.set_xticks(x);  ax2.set_xticklabels(bs_vals)
ax2.legend();  ax2.grid(axis="y", alpha=0.4)

fig.suptitle(
    f"ibm-granite/granite-embedding-97m-multilingual-r2  |  {N_TEXTS} variable-length texts",
    fontsize=11,
)
plt.tight_layout()
plt.savefig("collator_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to collator_benchmark.png")

## Results table

In [ ]:
header = f"{'Batch':>6}  {'Method':<15}  {'Throughput (s/s)':>17}  {'Latency (ms/s)':>15}  {'Speedup':>8}"
print(header)
print("-" * len(header))
for bs in BATCH_SIZES:
    s = next(r for r in results if r["label"] == "Standard ST" and r["batch_size"] == bs)
    c = next(r for r in results if r["label"] == "Collator ST" and r["batch_size"] == bs)
    sp = c["throughput"] / s["throughput"]
    for r, tag in [(s, ""), (c, f"{sp:+.0%}")]:
        print(f"{r['batch_size']:>6}  {r['label']:<15}  {r['throughput']:>17.1f}  "
              f"{r['latency_ms']:>15.2f}  {tag:>8}")
    print()

## Key design notes

### What makes ModernBERT special here
ModernBERT's `forward` signature exposes `cu_seqlens`, `max_seqlen`, `indices`, `batch_size`, and `seq_len`
as first-class parameters — it was designed with varlen flash attention as a first-class citizen.  
In varlen mode it expects **1-D `input_ids`** (the flat unpadded token sequence) so that
`hidden_states.shape[0] == total_tokens` matches the shape that `flash_attn_varlen_qkvpacked_func` returns.

### Extending to other architectures
For models like BERT/RoBERTa that do **not** expose `cu_seqlens` as a top-level argument,
the safest option is the **standard padded path** (`SentenceTransformer.encode` as-is).  
The collator-based approach requires the model to natively route `cu_seqlens` down to its attention layers.

### When is the speedup largest?
The gain is proportional to padding waste:
- **High gain**: variable-length corpora (queries + passages mixed), long-tail length distributions.
- **Low gain**: uniform-length inputs (already near-zero padding even with the standard path).

### Relationship to `benchmark_embedding_timing.py`
`CollatorGPUEmbedder` in `scripts/speed_test/benchmark_embedding_timing.py` implements the same idea
on top of a raw HuggingFace `AutoModel`, without the `SentenceTransformer` wrapper.
`CollatorSentenceTransformer` provides the same optimisation while preserving the familiar ST API.